# Exp0.3 — Halo Refinement + Interior HF Diagnostic

## Контекст
Exp0.2 показал:
- **MSE_rgb/PSNR**: adaptive выигрывает 98–100% изображений → ядро гипотезы подтверждено.
- **MSE_highfreq**: adaptive *проигрывает* baseline при малом K → шовный артефакт на границах refined/unrefined тайлов.

## Два эксперимента

**Exp A — Halo-refine:** вместо жёсткой подмены тайла — blending через weight mask с cosine feather.
Гипотеза: MSE_highfreq перестанет деградировать на малых K.

**Exp B — Interior-only HF метрика:** Лапласиан считается только внутри тайлов (исключая 1px у границ).
Если adaptive выигрывает interior HF на малых K — проблема именно в швах, не в выборе.

## K_list
Явный: `[1, 2, 4, 8, 16]` для 64 тайлов — без дубликатов coverage.

## CIFAR
Тот же `./data/` что и Exp0.2.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, convolve
import warnings, time, pathlib, json

warnings.filterwarnings("ignore")

# ═════════════ CONFIG ═════════════
H = 128
W = 128
tile_size = 16
downsample_factor = 4
alpha = 0.25
n_iter = 8

K_list = [1, 2, 4, 8, 16]

N_images_debug = 128
N_images_final = 2000
mode = "debug"  # "debug" or "final"

halo_widths = [2, 3, 4]

edge_eps = 1e-6
max_static_images = 8

LAPLACIAN_K = np.array([[0, -1, 0],
                        [-1,  4, -1],
                        [0, -1, 0]], dtype=np.float32)

# Derived
N_images = N_images_debug if mode == "debug" else N_images_final
tiles_y = H // tile_size
tiles_x = W // tile_size
N_tiles = tiles_y * tiles_x

OUT_DIR = pathlib.Path("out/exp03")
OUT_DIR.mkdir(parents=True, exist_ok=True)

config = dict(H=H, W=W, tile_size=tile_size, downsample_factor=downsample_factor,
              alpha=alpha, n_iter=n_iter, K_list=K_list,
              halo_widths=halo_widths, N_images=N_images, mode=mode)
with open(OUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Mode: {mode}, N_images: {N_images}")
print(f"Grid: {tiles_y}x{tiles_x} = {N_tiles} tiles")
print(f"K_list: {K_list} (coverage: {[f'{k/N_tiles:.1%}' for k in K_list]})")
print(f"Halo widths: {halo_widths}")


In [ ]:
def _load_cifar_torchvision():
    import torchvision
    ds = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
    return ds.data.astype(np.float32) / 255.0

def _load_cifar_keras():
    from keras.datasets import cifar10
    (x_train, _), (_, _) = cifar10.load_data()
    return x_train.astype(np.float32) / 255.0

def _load_cifar_pickle(path="./data/cifar-10-python.tar.gz"):
    import tarfile, pickle
    data_batches = []
    with tarfile.open(path, "r:gz") as tar:
        for member in tar.getmembers():
            if "data_batch" in member.name:
                f = tar.extractfile(member)
                batch = pickle.load(f, encoding="bytes")
                arr = batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
                data_batches.append(arr)
    return np.concatenate(data_batches, axis=0).astype(np.float32) / 255.0

cifar_raw = None
for name, fn in [("torchvision", _load_cifar_torchvision),
                 ("keras", _load_cifar_keras),
                 ("local pickle", lambda: _load_cifar_pickle())]:
    try:
        print(f"Trying {name}...", end=" ")
        cifar_raw = fn()
        print(f"OK! {cifar_raw.shape}")
        break
    except Exception as e:
        print(f"Failed ({e})")

if cifar_raw is None:
    raise RuntimeError(
        "CIFAR not found. Run: wget https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -P ./data/")

print(f"Loaded: {cifar_raw.shape}, [{cifar_raw.min():.2f}, {cifar_raw.max():.2f}]")


In [ ]:
def upscale_image(img32, th, tw):
    up = zoom(img32, (th/img32.shape[0], tw/img32.shape[1], 1), order=1)
    return np.clip(up[:th, :tw, :], 0, 1).astype(np.float32)

def coarse_from_image(img, s):
    h, w, c = img.shape
    small = zoom(img, (1.0/s, 1.0/s, 1), order=1)
    big = zoom(small, (s, s, 1), order=1)
    return np.clip(big[:h, :w, :], 0, 1).astype(np.float32)

rng_data = np.random.RandomState(0)
indices = rng_data.choice(len(cifar_raw), size=N_images, replace=False)
indices.sort()

print(f"Upscaling {N_images} images to {H}x{W}...")
t0 = time.time()
images = np.stack([upscale_image(cifar_raw[i], H, W) for i in indices])
print(f"Done in {time.time()-t0:.1f}s")

print("Computing coarse...")
t0 = time.time()
coarses = np.stack([coarse_from_image(images[i], downsample_factor) for i in range(N_images)])
print(f"Done in {time.time()-t0:.1f}s, mean coarse MSE: {np.mean((images-coarses)**2):.6f}")


In [ ]:
def to_grayscale(img):
    return (0.299*img[:,:,0] + 0.587*img[:,:,1] + 0.114*img[:,:,2]).astype(np.float32)

def compute_tile_rho(img, coarse, tile_size):
    Y_gt = to_grayscale(img)
    Y_co = to_grayscale(coarse)
    residual = Y_gt - Y_co
    ty, tx = Y_gt.shape[0]//tile_size, Y_gt.shape[1]//tile_size
    rho = np.zeros((ty, tx), dtype=np.float32)
    for i in range(ty):
        for j in range(tx):
            y0, y1 = i*tile_size, (i+1)*tile_size
            x0, x1 = j*tile_size, (j+1)*tile_size
            rho[i,j] = np.mean(residual[y0:y1, x0:x1]**2)
    return rho

def select_tiles_adaptive(rho, K):
    ty, tx = rho.shape
    entries = [(-rho[i,j], i*tx+j) for i in range(ty) for j in range(tx)]
    entries.sort()
    return [(fi//tx, fi%tx) for _, fi in entries[:K]]

def select_tiles_random(rho, K, rng):
    ty, tx = rho.shape
    all_t = [(i,j) for i in range(ty) for j in range(tx)]
    idxs = rng.choice(len(all_t), size=min(K, len(all_t)), replace=False)
    return [all_t[idx] for idx in sorted(idxs)]

print("Tile functions ready.")


In [ ]:
def refine_hard(pred, GT, selected_tiles, tile_size, n_iter, alpha):
    out = pred.copy()
    for _ in range(n_iter):
        for (i, j) in selected_tiles:
            y0, y1 = i*tile_size, (i+1)*tile_size
            x0, x1 = j*tile_size, (j+1)*tile_size
            out[y0:y1, x0:x1, :] += alpha * (GT[y0:y1, x0:x1, :] - out[y0:y1, x0:x1, :])
    return out

def make_halo_mask(tile_size, halo_width):
    """2D weight mask: 1 inside, cosine taper to 0 at edges over halo_width pixels."""
    w = np.ones((tile_size, tile_size), dtype=np.float32)
    hw = halo_width
    for d in range(hw):
        val = 0.5 * (1 - np.cos(np.pi * (d + 0.5) / hw))
        w[d, :] = np.minimum(w[d, :], val)
        w[tile_size-1-d, :] = np.minimum(w[tile_size-1-d, :], val)
        w[:, d] = np.minimum(w[:, d], val)
        w[:, tile_size-1-d] = np.minimum(w[:, tile_size-1-d], val)
    return w

def refine_halo(pred, GT, selected_tiles, tile_size, n_iter, alpha, halo_mask):
    """Refine selected tiles, then blend with coarse via halo_mask.
    out_tile = mask * refined_tile + (1 - mask) * coarse_tile."""
    coarse_backup = pred.copy()
    refined = pred.copy()
    for _ in range(n_iter):
        for (i, j) in selected_tiles:
            y0, y1 = i*tile_size, (i+1)*tile_size
            x0, x1 = j*tile_size, (j+1)*tile_size
            refined[y0:y1, x0:x1, :] += alpha * (GT[y0:y1, x0:x1, :] - refined[y0:y1, x0:x1, :])
    out = coarse_backup.copy()
    w = halo_mask[:, :, np.newaxis]
    for (i, j) in selected_tiles:
        y0, y1 = i*tile_size, (i+1)*tile_size
        x0, x1 = j*tile_size, (j+1)*tile_size
        out[y0:y1, x0:x1, :] = w * refined[y0:y1, x0:x1, :] + (1 - w) * coarse_backup[y0:y1, x0:x1, :]
    return out

# Precompute halo masks
halo_masks = {hw: make_halo_mask(tile_size, hw) for hw in halo_widths}

# Visualize
fig, axes = plt.subplots(1, len(halo_widths), figsize=(4*len(halo_widths), 3.5))
if len(halo_widths) == 1: axes = [axes]
for ax, hw in zip(axes, halo_widths):
    m = halo_masks[hw]
    ax.imshow(m, cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"halo={hw}, center={m[tile_size//2,tile_size//2]:.2f}, corner={m[0,0]:.3f}", fontsize=10)
    ax.axis("off")
plt.suptitle("Halo weight masks", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
def _make_interior_mask(h, w, tile_size, border=1):
    """True for pixels >= border px from any tile boundary."""
    mask = np.ones((h, w), dtype=bool)
    for i in range(1, h // tile_size):
        y = i * tile_size
        for d in range(border):
            if y-1-d >= 0: mask[y-1-d, :] = False
            if y+d < h:    mask[y+d, :] = False
    for j in range(1, w // tile_size):
        x = j * tile_size
        for d in range(border):
            if x-1-d >= 0: mask[:, x-1-d] = False
            if x+d < w:    mask[:, x+d] = False
    return mask

INTERIOR_MASK = _make_interior_mask(H, W, tile_size, border=1)
print(f"Interior mask: {INTERIOR_MASK.sum()}/{INTERIOR_MASK.size} px ({INTERIOR_MASK.mean():.1%} kept)")

def compute_metrics(img_gt, img_pred):
    mse_rgb = float(np.mean((img_gt - img_pred)**2))
    psnr_rgb = float(10 * np.log10(1.0 / max(mse_rgb, 1e-12)))

    Y_gt = to_grayscale(img_gt)
    Y_pred = to_grayscale(img_pred)
    L_gt = np.abs(convolve(Y_gt, LAPLACIAN_K, mode='reflect'))
    L_pred = np.abs(convolve(Y_pred, LAPLACIAN_K, mode='reflect'))
    norm = np.mean(L_gt) + edge_eps
    L_gt_n = L_gt / norm
    L_pred_n = L_pred / norm
    diff_hf = (L_pred_n - L_gt_n)**2

    return {
        "MSE_rgb": mse_rgb,
        "PSNR_rgb": psnr_rgb,
        "MSE_highfreq": float(np.mean(diff_hf)),
        "MSE_hf_interior": float(np.mean(diff_hf[INTERIOR_MASK])),
    }

m = compute_metrics(images[0], coarses[0])
print(f"Test: { {k: f'{v:.6f}' for k,v in m.items()} }")


In [ ]:
results = []
t_start = time.time()

for img_idx in range(N_images):
    img = images[img_idx]
    coarse = coarses[img_idx]
    rho = compute_tile_rho(img, coarse, tile_size)

    # Coarse reference
    m_c = compute_metrics(img, coarse)
    results.append({"image_idx": img_idx, "K": 0, "method": "coarse", **m_c})

    for K in K_list:
        sel_a = select_tiles_adaptive(rho, K)
        rng_bl = np.random.RandomState(img_idx * 10000 + K)
        sel_b = select_tiles_random(rho, K, rng_bl)

        # Baseline hard
        pred_bl = refine_hard(coarse.copy(), img, sel_b, tile_size, n_iter, alpha)
        m_bl = compute_metrics(img, pred_bl)
        results.append({"image_idx": img_idx, "K": K, "method": "baseline_hard", **m_bl})

        # Adaptive hard
        pred_ah = refine_hard(coarse.copy(), img, sel_a, tile_size, n_iter, alpha)
        m_ah = compute_metrics(img, pred_ah)
        results.append({"image_idx": img_idx, "K": K, "method": "adaptive_hard", **m_ah})

        # Adaptive halo (each width)
        for hw in halo_widths:
            pred_h = refine_halo(coarse.copy(), img, sel_a, tile_size, n_iter, alpha, halo_masks[hw])
            m_h = compute_metrics(img, pred_h)
            results.append({"image_idx": img_idx, "K": K, "method": f"adaptive_halo_{hw}", **m_h})

    if (img_idx+1) % max(1, N_images//10) == 0:
        print(f"  {img_idx+1}/{N_images}...")

elapsed = time.time() - t_start
df = pd.DataFrame(results)
print(f"\nDone in {elapsed:.1f}s - {len(df)} rows, methods: {sorted(df.method.unique())}")
df.head()


In [ ]:
showcase_K = 4
showcase_hw = halo_widths[len(halo_widths)//2]
n_show = min(max_static_images // 2, N_images)
showcase_idxs = np.linspace(0, N_images-1, n_show, dtype=int)

for img_idx in showcase_idxs:
    img = images[img_idx]
    coarse = coarses[img_idx]
    rho = compute_tile_rho(img, coarse, tile_size)

    sel_a = select_tiles_adaptive(rho, showcase_K)
    rng_bl = np.random.RandomState(img_idx * 10000 + showcase_K)
    sel_b = select_tiles_random(rho, showcase_K, rng_bl)

    pred_bl = refine_hard(coarse.copy(), img, sel_b, tile_size, n_iter, alpha)
    pred_ah = refine_hard(coarse.copy(), img, sel_a, tile_size, n_iter, alpha)
    pred_halo = refine_halo(coarse.copy(), img, sel_a, tile_size, n_iter, alpha, halo_masks[showcase_hw])

    err_c  = np.sqrt(np.mean((img - coarse)**2, axis=2))
    err_bl = np.sqrt(np.mean((img - pred_bl)**2, axis=2))
    err_ah = np.sqrt(np.mean((img - pred_ah)**2, axis=2))
    err_hl = np.sqrt(np.mean((img - pred_halo)**2, axis=2))
    emax = max(err_c.max(), err_bl.max(), err_ah.max(), err_hl.max(), 1e-6)

    fig, axes = plt.subplots(2, 5, figsize=(22, 9))
    fig.suptitle(f"Image #{img_idx} | K={showcase_K} | halo={showcase_hw}", fontsize=13)

    for ax, (t, d) in zip(axes[0], [("GT", img), ("Coarse", coarse), ("Baseline", pred_bl),
                                      ("Adaptive hard", pred_ah), ("Adaptive halo", pred_halo)]):
        ax.imshow(np.clip(d, 0, 1)); ax.set_title(t, fontsize=10); ax.axis("off")

    for ax, (t, d) in zip(axes[1], [("|err| coarse", err_c), ("|err| baseline", err_bl),
                                      ("|err| adapt hard", err_ah), ("|err| adapt halo", err_hl),
                                      ("rho heatmap", rho)]):
        cm = "hot" if "rho" in t else "magma"
        vmax = None if "rho" in t else emax
        ax.imshow(d, cmap=cm, vmin=0, vmax=vmax, interpolation="nearest" if "rho" in t else "bilinear")
        ax.set_title(t, fontsize=10); ax.axis("off")

    plt.tight_layout(); plt.show()

print(f"Showcase: {len(showcase_idxs)} images")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5.5))
fig.suptitle("Exp A: Effect of Halo on All Metrics (mean +/- std over dataset)", fontsize=14)

metrics_plot = ["MSE_rgb", "MSE_highfreq", "MSE_hf_interior", "PSNR_rgb"]

style = {
    "baseline_hard":   ("tab:orange", "s", "--", "baseline hard"),
    "adaptive_hard":   ("tab:red",    "^", "-",  "adaptive hard"),
}
hcolors = {2: "tab:blue", 3: "tab:green", 4: "tab:purple"}
for hw in halo_widths:
    style[f"adaptive_halo_{hw}"] = (hcolors.get(hw,"tab:cyan"), "o", "-", f"adaptive halo={hw}")

for ax, metric in zip(axes, metrics_plot):
    cv = df[df.method=="coarse"][metric].mean()
    ax.axhline(cv, color="gray", ls="--", lw=1, label="coarse")

    for method, (color, marker, ls, label) in style.items():
        sub = df[df.method == method]
        if sub.empty: continue
        grp = sub.groupby("K")[metric]
        ax.errorbar(grp.mean().index, grp.mean().values, yerr=grp.std().values,
                    color=color, marker=marker, ls=ls, label=label, capsize=2, ms=5)

    ax.set_xlabel("K (tiles)"); ax.set_ylabel(metric); ax.set_title(metric)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Exp B: Full HF vs Interior-only HF (adaptive_hard vs baseline_hard)", fontsize=14)

for ax, metric, title in zip(axes,
    ["MSE_highfreq", "MSE_hf_interior"],
    ["MSE_highfreq (all pixels)", "MSE_hf_interior (excl. tile boundaries)"]):

    cv = df[df.method=="coarse"][metric].mean()
    ax.axhline(cv, color="gray", ls="--", lw=1, label="coarse")

    for method, color, marker in [("adaptive_hard","tab:blue","o"), ("baseline_hard","tab:orange","s")]:
        grp = df[df.method==method].groupby("K")[metric]
        ax.errorbar(grp.mean().index, grp.mean().values, yerr=grp.std().values,
                    color=color, marker=marker, label=method.replace("_"," "), capsize=3)

    ax.set_xlabel("K (tiles)"); ax.set_ylabel(metric); ax.set_title(title)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
print("=== PER-IMAGE WIN RATES (vs baseline_hard) ===\n")

for metric in ["MSE_rgb", "MSE_highfreq", "MSE_hf_interior"]:
    print(f"--- {metric} (lower = better) ---")
    for K in K_list:
        sub = df[df.K == K]
        bl = sub[sub.method=="baseline_hard"].set_index("image_idx")[metric]
        parts = []
        for method in ["adaptive_hard"] + [f"adaptive_halo_{hw}" for hw in halo_widths]:
            ad = sub[sub.method==method].set_index("image_idx")[metric]
            common = bl.index.intersection(ad.index)
            wr = (ad.loc[common] < bl.loc[common]).mean() * 100
            parts.append(f"{method.replace('adaptive_',''):>8s}: {wr:5.1f}%")
        print(f"  K={K:2d}  " + "  |  ".join(parts))
    print()


In [ ]:
# ═════════════ SUMMARY + VERDICT ═════════════

summary_rows = []
for K in K_list:
    sub = df[df.K == K]
    bl = sub[sub.method == "baseline_hard"]
    for method in ["adaptive_hard"] + [f"adaptive_halo_{hw}" for hw in halo_widths]:
        ad = sub[sub.method == method]
        for metric in ["MSE_rgb", "PSNR_rgb", "MSE_highfreq", "MSE_hf_interior"]:
            a_mean = ad[metric].mean()
            b_mean = bl[metric].mean()
            if metric == "PSNR_rgb":
                wins = a_mean > b_mean
                delta = (a_mean - b_mean) / max(abs(b_mean), 1e-12) * 100
            else:
                wins = a_mean < b_mean
                delta = (b_mean - a_mean) / max(abs(b_mean), 1e-12) * 100
            summary_rows.append({
                "K": K, "method": method, "metric": metric,
                "adaptive_mean": round(a_mean, 7), "baseline_mean": round(b_mean, 7),
                "delta_%": round(delta, 2), "adaptive_wins": wins,
            })

summary = pd.DataFrame(summary_rows)

# ── Print key findings ──
print("=" * 70)
print("KEY FINDINGS")
print("=" * 70)

# 1. MSE_rgb: should always win
mse_rgb_wins = summary[(summary.metric=="MSE_rgb") & (summary.method=="adaptive_hard")]
print(f"\n1) MSE_rgb (adaptive_hard vs baseline):")
print(f"   Wins: {mse_rgb_wins.adaptive_wins.sum()}/{len(mse_rgb_wins)}")
for _, r in mse_rgb_wins.iterrows():
    print(f"   K={r.K:2d}: delta={r['delta_%']:+.2f}%")

# 2. MSE_highfreq: hard vs halo
print(f"\n2) MSE_highfreq — does halo fix the degradation?")
for method in ["adaptive_hard"] + [f"adaptive_halo_{hw}" for hw in halo_widths]:
    hf = summary[(summary.metric=="MSE_highfreq") & (summary.method==method)]
    w = hf.adaptive_wins.sum()
    print(f"   {method:>20s}: wins {w}/{len(hf)} " +
          "  ".join(f"K={r.K}:{r['delta_%']:+.1f}%" for _,r in hf.iterrows()))

# 3. Interior HF: proves seam hypothesis
print(f"\n3) MSE_hf_interior (adaptive_hard, excl. boundaries):")
hfi = summary[(summary.metric=="MSE_hf_interior") & (summary.method=="adaptive_hard")]
w = hfi.adaptive_wins.sum()
print(f"   Wins: {w}/{len(hfi)}")
for _, r in hfi.iterrows():
    print(f"   K={r.K:2d}: delta={r['delta_%']:+.2f}%")

# ── Verdicts ──
print(f"\n{'='*70}")
print("VERDICTS")
print("=" * 70)

# Exp A verdict
best_halo = None
best_hf_wins = 0
for hw in halo_widths:
    hf = summary[(summary.metric=="MSE_highfreq") & (summary.method==f"adaptive_halo_{hw}")]
    w = int(hf.adaptive_wins.sum())
    if w > best_hf_wins:
        best_hf_wins = w
        best_halo = hw

hf_hard = summary[(summary.metric=="MSE_highfreq") & (summary.method=="adaptive_hard")]
hard_hf_wins = int(hf_hard.adaptive_wins.sum())

if best_hf_wins > hard_hf_wins:
    print(f"\nExp A: \u2705 Halo HELPS — best halo={best_halo} wins {best_hf_wins}/{len(K_list)} "
          f"vs hard {hard_hf_wins}/{len(K_list)} on MSE_highfreq")
else:
    print(f"\nExp A: \u274c Halo does NOT help MSE_highfreq "
          f"(best halo wins {best_hf_wins}/{len(K_list)}, hard wins {hard_hf_wins}/{len(K_list)})")

# Exp B verdict
interior_wins = int(hfi.adaptive_wins.sum())
if interior_wins > hard_hf_wins:
    print(f"\nExp B: \u2705 Interior HF confirms SEAM HYPOTHESIS — "
          f"adaptive wins {interior_wins}/{len(K_list)} inside tiles "
          f"vs {hard_hf_wins}/{len(K_list)} with boundaries included")
else:
    print(f"\nExp B: \u274c Interior HF does NOT differentiate ({interior_wins} vs {hard_hf_wins})")

print(f"\n{'='*70}")


In [ ]:
df.to_csv(OUT_DIR / "results.csv", index=False)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
print(f"Saved: {OUT_DIR / 'results.csv'}, {OUT_DIR / 'summary.csv'}")


In [ ]:
print("Running repro-check...")
for run in range(2):
    img_r = images[0]; coarse_r = coarses[0]
    rho_r = compute_tile_rho(img_r, coarse_r, tile_size)
    sel_r = select_tiles_adaptive(rho_r, 4)
    pred_r = refine_hard(coarse_r.copy(), img_r, sel_r, tile_size, n_iter, alpha)
    pred_h = refine_halo(coarse_r.copy(), img_r, sel_r, tile_size, n_iter, alpha, halo_masks[3])
    m_hard = compute_metrics(img_r, pred_r)
    m_halo = compute_metrics(img_r, pred_h)
    if run == 0:
        sel_0, mh_0, mhl_0 = sel_r, m_hard, m_halo
    else:
        assert sel_r == sel_0, "REPRO FAIL: selection"
        for k in mh_0:
            assert abs(mh_0[k] - m_hard[k]) < 1e-10, f"REPRO FAIL hard: {k}"
            assert abs(mhl_0[k] - m_halo[k]) < 1e-10, f"REPRO FAIL halo: {k}"

print("\u2705 Repro-check PASSED (hard + halo)")
